# Integration + Unified Results

**This notebook:**
1. Loads the trained PPO agent and VolLSTM from checkpoints
2. Creates an RL hedging env that uses LSTM-forecast vol at each step
3. Compares: (a) RL with constant vol vs (b) RL with LSTM vol
4. Produces the unified report plots

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path('..') / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from stable_baselines3 import PPO
from env import HedgingEnv
from lstm_model import VolLSTM, predict
from data_pipeline import load_nifty, make_sequences, split_data
from backtest import evaluate_rl_agent, plot_pnl_comparison
from bsm import bsm_price

## 1. Load checkpoints

In [ ]:
# RL model
rl_model = PPO.load('../models/ppo_hedge_v1')

# LSTM model
lstm = VolLSTM(input_size=4, hidden1=64, hidden2=32)
lstm.load_state_dict(torch.load('../models/lstm_vol_best.pt', map_location='cpu'))
lstm.eval()

print('Both models loaded.')

## 2. Build LSTM vol schedules for test-set episodes

In [ ]:
df_nifty = load_nifty()
X_all, y_all, dates_all = make_sequences(df_nifty, seq_len=60)
splits = split_data(X_all, y_all, dates_all)

X_test = splits['test']['X']
lstm_vols_test = predict(lstm, X_test)

print(f'LSTM vol forecasts: min={lstm_vols_test.min():.3f}, "
      f"max={lstm_vols_test.max():.3f}, mean={lstm_vols_test.mean():.3f}')

## 3. Integration: RL agent with LSTM vol schedule

We run the pre-trained PPO agent (trained with constant σ=0.20) on environments
where the GBM vol at each step is drawn from the LSTM forecasts.
This tests whether the agent generalises to time-varying vol.

In [ ]:
ENV_PARAMS_BASE = dict(S0=100, K=100, T=21/252, r=0.05,
                       sigma=0.20, n_steps=21, kappa=0.001)

N_EPS = 500

# --- (a) constant vol env (baseline) ---
results_const = evaluate_rl_agent(rl_model, HedgingEnv, ENV_PARAMS_BASE, n_episodes=N_EPS)

# --- (b) LSTM vol schedule ---
# Sample a random 21-step window from the LSTM forecasts for each episode
rng = np.random.default_rng(0)

def make_lstm_params(lstm_vols):
    n_test = len(lstm_vols)
    start  = rng.integers(0, n_test - 21)
    schedule = lstm_vols[start : start + 21]
    return {**ENV_PARAMS_BASE, 'sigma_schedule': schedule}

terminal_pnls_lstm = []
episode_tcs_lstm   = []

for _ in range(N_EPS):
    params = make_lstm_params(lstm_vols_test)
    env = HedgingEnv(**params)
    obs, _ = env.reset()
    done = False
    while not done:
        action, _ = rl_model.predict(obs, deterministic=True)
        obs, _, terminated, truncated, info = env.step(action)
        done = terminated or truncated
    terminal_pnls_lstm.append(info.get('terminal_pnl', np.nan))
    episode_tcs_lstm.append(info.get('episode_tc', np.nan))

pnl_lstm = np.array(terminal_pnls_lstm)
print('Constant vol:  MAE={:.4f}'.format(np.nanmean(np.abs(results_const['terminal_pnl']))))
print('LSTM vol sched: MAE={:.4f}'.format(np.nanmean(np.abs(pnl_lstm))))

## 4. Unified results plot

In [ ]:
plot_pnl_comparison({
    'RL — constant σ':     results_const['terminal_pnl'],
    'RL — LSTM σ schedule': pnl_lstm,
}, title='Integration: RL agent under constant vs LSTM-forecast vol')

In [ ]:
# Summary table
summary = pd.DataFrame({
    'Setup':     ['RL constant σ', 'RL LSTM σ'],
    'Mean PnL':  [np.nanmean(results_const['terminal_pnl']), np.nanmean(pnl_lstm)],
    'Std PnL':   [np.nanstd(results_const['terminal_pnl']),  np.nanstd(pnl_lstm)],
    'MAE':       [np.nanmean(np.abs(results_const['terminal_pnl'])),
                  np.nanmean(np.abs(pnl_lstm))],
    'Mean TC':   [np.nanmean(results_const['episode_tc']),
                  np.nanmean(episode_tcs_lstm)],
}).set_index('Setup').round(5)

print(summary)
summary.to_csv('../results/integration_summary.csv')

## 5. Full report: all plots in one grid

In [ ]:
import matplotlib.image as mpimg
from pathlib import Path

results_dir = Path('../results')
plot_files = [
    ('vol_forecast.png',        'LSTM Vol Forecast'),
    ('lstm_scatter.png',        'Predicted vs Actual Vol'),
    ('vega_surface.png',        'Vega Sensitivity'),
    ('pricing_error_bar.png',   'Pricing Horse Race'),
    ('pnl_comparison.png',      'Terminal PnL Distributions'),
    ('hedge_ratios.png',        'Hedge Ratio Paths'),
    ('cost_comparison.png',     'Transaction Costs'),
    ('lstm_vs_implied.png',     'LSTM vs Implied Vol'),
]

available = [(p, t) for p, t in plot_files if (results_dir / p).exists()]

if available:
    n = len(available)
    cols = 4
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols*5, rows*4))
    axes = axes.flatten()
    for ax, (fname, title) in zip(axes, available):
        img = mpimg.imread(results_dir / fname)
        ax.imshow(img)
        ax.set_title(title, fontsize=10)
        ax.axis('off')
    for ax in axes[len(available):]:
        ax.axis('off')
    plt.suptitle('Full Report — Projects 1 & 3', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(results_dir / 'report_grid.png', dpi=120)
    plt.show()
else:
    print('Run p1_train.ipynb and p3_train.ipynb first to generate plots.')